In [1]:
import sys
import os
import pandas as pd

os.chdir(os.path.abspath(".."))
sys.path.append(os.path.abspath(".."))

In [4]:
from dotenv import load_dotenv
import requests, json
load_dotenv()


email = "YOUR_EMAIL"      # J-Quants登録メール
password = "YOUR_PASS"    # パスワード
auth_payload = {"mailaddress": email, "password": password}
r = requests.post("https://api.jquants.com/v1/token/auth_user",
                  data=json.dumps(auth_payload))
refresh_token = os.getenv('JPX_API_KEY')

In [5]:
id_url = f"https://api.jquants.com/v1/token/auth_refresh?refreshtoken={refresh_token}"
r2 = requests.post(id_url)
id_token = r2.json().get("idToken")
print("ID Token:", id_token[:20], "...")  # トークンは長い文字列

ID Token: eyJraWQiOiJHQXNvU2xx ...


# J-Quants Standard API で日次株価を取得し Postgres に保存  
TOPIX500＋グロース市場を含む全銘柄（companies テーブルに登録済み）の  
2016‑01‑01 以降の日次株価を取得して **daily_quotes** テーブルへ UPSERT  

In [6]:
import os, json, time, logging
from datetime import datetime
from typing import Optional, List

import pandas as pd
import requests
from sqlalchemy import create_engine, text
from dotenv import load_dotenv

# ───────────── 環境変数読み込み ─────────────
load_dotenv()                         # .env を読み込む
JQ_USER   = os.getenv("JQUANTS_USER")
JQ_PASS   = os.getenv("JQUANTS_PASS")
JQ_TOKEN  = os.getenv("JPX_API_KEY")  # ← リフレッシュトークン
DB_USER   = os.getenv("POSTGRES_USER")
DB_PASS   = os.getenv("POSTGRES_PASSWORD")
DB_HOST   = os.getenv("POSTGRES_HOST", "localhost")
DB_PORT   = os.getenv("POSTGRES_PORT", "5432")
DB_NAME   = os.getenv("POSTGRES_DB")

# ───────────── ロギング ─────────────
logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)

# ───────────── DB エンジン ─────────────
db_url = f"postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(db_url)

In [7]:
ddl = """
CREATE TABLE IF NOT EXISTS daily_quotes (
    code              VARCHAR(5)  NOT NULL,
    date              DATE        NOT NULL,
    open              NUMERIC(10,2),
    high              NUMERIC(10,2),
    low               NUMERIC(10,2),
    close             NUMERIC(10,2),
    volume            BIGINT,
    turnover_value    BIGINT,
    adjustment_factor NUMERIC(9,6),
    adjusted_open     NUMERIC(10,2),
    adjusted_high     NUMERIC(10,2),
    adjusted_low      NUMERIC(10,2),
    adjusted_close    NUMERIC(10,2),
    adjusted_volume   BIGINT,
    upper_limit       BOOLEAN,
    lower_limit       BOOLEAN,
    PRIMARY KEY (code, date)
);
"""
with engine.begin() as conn:
    conn.execute(text(ddl))
logger.info("✅ daily_quotes table is ready")

2025-07-02 23:03:05,080 [INFO] ✅ daily_quotes table is ready


In [8]:
## 認証ヘルパ

# def get_id_token() -> str:
#     """環境変数にリフレッシュトークンがあればそれを使用、無ければメール/パスで取得"""
#     if JQ_TOKEN:                                  # リフレッシュトークン → ID トークン
#         resp = requests.post(
#             "https://api.jquants.com/v1/token/auth_refresh",
#             params={"refreshtoken": JQ_TOKEN}
#         )
#         resp.raise_for_status()
#         return resp.json()["idToken"]

#     # メール&パスワード認証 → リフレッシュトークン → ID トークン
#     auth_payload = {"mailaddress": JQ_USER, "password": JQ_PASS}
#     r = requests.post(
#         "https://api.jquants.com/v1/token/auth_user",
#         data=json.dumps(auth_payload),
#         headers={"Content-Type": "application/json"}
#     )
#     r.raise_for_status()
#     refresh = r.json()["refreshToken"]
#     r2 = requests.post(
#         "https://api.jquants.com/v1/token/auth_refresh",
#         params={"refreshtoken": refresh}
#     )
#     r2.raise_for_status()
#     return r2.json()["idToken"]

# ID_TOKEN = get_id_token()
# logger.info("✅ J‑Quants ID token acquired")
ID_TOKEN = id_token

In [9]:
## 株価取得関数(ページネーション対応)

def fetch_daily_quotes(code: str,
                       date_from: str,
                       date_to: str,
                       token: str,
                       max_retry: int = 3) -> pd.DataFrame | None:
    """指定コード・期間の株価 DataFrame（空なら None）"""
    url = "https://api.jquants.com/v1/prices/daily_quotes"
    headers = {"Authorization": f"Bearer {token}"}
    params = {"code": code, "from": date_from, "to": date_to}

    all_rows: List[dict] = []
    retry_cnt = 0
    while True:
        try:
            res = requests.get(url, headers=headers, params=params, timeout=20)
            if res.status_code == 401 and retry_cnt < max_retry:  # トークン失効 → 再取得
                logger.warning("ID token expired, refreshing...")
                token = get_id_token()
                headers["Authorization"] = f"Bearer {token}"
                retry_cnt += 1
                continue
            res.raise_for_status()
        except Exception as e:
            logger.error(f"[ERROR] code={code} – {e}")
            return None

        js = res.json()
        all_rows.extend(js.get("daily_quotes", []))

        if "pagination_key" not in js:
            break
        params["pagination_key"] = js["pagination_key"]

    if not all_rows:
        return None

    df = pd.DataFrame(all_rows)
    # 型変換・列名統一
    num_cols = ["Open","High","Low","Close","Volume","TurnoverValue",
                "AdjustmentFactor","AdjustmentOpen","AdjustmentHigh",
                "AdjustmentLow","AdjustmentClose","AdjustmentVolume"]
    for c in num_cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    df.rename(columns={
        "Code": "code", "Date": "date",
        "Open": "open", "High": "high", "Low": "low",
        "Close": "close", "Volume": "volume",
        "TurnoverValue": "turnover_value",
        "AdjustmentFactor": "adjustment_factor",
        "AdjustmentOpen": "adjusted_open",
        "AdjustmentHigh": "adjusted_high",
        "AdjustmentLow": "adjusted_low",
        "AdjustmentClose": "adjusted_close",
        "AdjustmentVolume": "adjusted_volume",
        "UpperLimit": "upper_limit",
        "LowerLimit": "lower_limit"
    }, inplace=True)

    # 日付型へ
    df["date"] = pd.to_datetime(df["date"]).dt.date
    return df

In [10]:
### companiesから対象証券コードを取得

# 4桁または5桁数字のみを抽出
sql_codes = """
    SELECT DISTINCT security_code
    FROM companies
"""
codes_df = pd.read_sql(sql_codes, engine)
codes = codes_df["security_code"].astype(str).tolist()
logger.info(f"対象銘柄数: {len(codes)}")

2025-07-02 23:03:15,255 [INFO] 対象銘柄数: 1264


In [13]:
import os
import logging
from datetime import datetime
import numpy as np
import pandas as pd
from tqdm import tqdm
from sqlalchemy import create_engine, text
from sqlalchemy.exc import DataError

# --- 環境変数読み込み & エンジン作成 ---
from dotenv import load_dotenv
load_dotenv()  # .env に POSTGRES_* が設定されている前提

DB_URL = (
    f"postgresql+psycopg2://{os.getenv('POSTGRES_USER')}:"
    f"{os.getenv('POSTGRES_PASSWORD')}@"
    f"{os.getenv('POSTGRES_HOST')}:{os.getenv('POSTGRES_PORT')}/"
    f"{os.getenv('POSTGRES_DB')}"
)
engine = create_engine(DB_URL)

# ロガー設定
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)

# --- パラメータ ---
DATE_FROM = "2016-01-01"
DATE_TO   = datetime.today().strftime("%Y-%m-%d")

# ここはあらかじめ取得している JPX 証券コードリスト
# e.g. codes = ['7203', '40550', …]
codes = [...]  

# UPSERT 用 SQL
insert_sql = text("""
INSERT INTO daily_quotes (
    code, date, open, high, low, close, volume, turnover_value,
    adjustment_factor, adjusted_open, adjusted_high, adjusted_low,
    adjusted_close, adjusted_volume, upper_limit, lower_limit
)
VALUES (
    :code, :date, :open, :high, :low, :close, :volume, :turnover_value,
    :adjustment_factor, :adjusted_open, :adjusted_high, :adjusted_low,
    :adjusted_close, :adjusted_volume, :upper_limit, :lower_limit
)
ON CONFLICT (code, date) DO UPDATE SET
    open              = EXCLUDED.open,
    high              = EXCLUDED.high,
    low               = EXCLUDED.low,
    close             = EXCLUDED.close,
    volume            = EXCLUDED.volume,
    turnover_value    = EXCLUDED.turnover_value,
    adjustment_factor = EXCLUDED.adjustment_factor,
    adjusted_open     = EXCLUDED.adjusted_open,
    adjusted_high     = EXCLUDED.adjusted_high,
    adjusted_low      = EXCLUDED.adjusted_low,
    adjusted_close    = EXCLUDED.adjusted_close,
    adjusted_volume   = EXCLUDED.adjusted_volume,
    upper_limit       = EXCLUDED.upper_limit,
    lower_limit       = EXCLUDED.lower_limit;
""")

# 件数カウント
success, fail = 0, 0
failed_codes = []

for code in tqdm(codes, desc="Upserting daily_quotes"):
    try:
        # API から取得（関数は環境に応じて定義済みとする）
        df = fetch_daily_quotes(code, DATE_FROM, DATE_TO, ID_TOKEN)
        if df is None or df.empty:
            logger.warning(f"[{code}] データなし → スキップ")
            fail += 1
            failed_codes.append(code)
            continue

        # NaN を None に置き換え（BIGINT 列への挿入エラー回避）
        for col in ["volume", "turnover_value", "adjusted_volume"]:
            df[col] = df[col].where(df[col].notna(), None).astype("Int64")

        # レコード化 & UPSERT
        records = df.to_dict("records")
        with engine.begin() as conn:
            conn.execute(insert_sql, records)

        success += 1
        logger.info(f"[{code}] {len(records)}件 upsert 成功")

    except DataError as de:
        logger.error(f"[{code}] DataError: {de}")
        fail += 1
        failed_codes.append(code)

    except Exception as e:
        logger.error(f"[{code}] その他エラー: {e}")
        fail += 1
        failed_codes.append(code)

# 最終ログ
logger.info(f"✅ 完了: 成功 {success} 銘柄, 失敗 {fail} 銘柄")
if failed_codes:
    logger.info(f"失敗銘柄一覧: {failed_codes}")


DataError: (psycopg2.errors.NumericValueOutOfRange) bigint out of range

[SQL: 
INSERT INTO daily_quotes (
    code, date, open, high, low, close, volume, turnover_value,
    adjustment_factor, adjusted_open, adjusted_high, adjusted_low,
    adjusted_close, adjusted_volume, upper_limit, lower_limit
)
VALUES (
    %(code)s, %(date)s, %(open)s, %(high)s, %(low)s, %(close)s, %(volume)s, %(turnover_value)s,
    %(adjustment_factor)s, %(adjusted_open)s, %(adjusted_high)s, %(adjusted_low)s,
    %(adjusted_close)s, %(adjusted_volume)s, %(upper_limit)s, %(lower_limit)s
)
ON CONFLICT (code, date) DO UPDATE SET
    open              = EXCLUDED.open,
    high              = EXCLUDED.high,
    low               = EXCLUDED.low,
    close             = EXCLUDED.close,
    volume            = EXCLUDED.volume,
    turnover_value    = EXCLUDED.turnover_value,
    adjustment_factor = EXCLUDED.adjustment_factor,
    adjusted_open     = EXCLUDED.adjusted_open,
    adjusted_high     = EXCLUDED.adjusted_hig

In [14]:
# 例: code='40550' だけを引いて最大値を調べる
from datetime import datetime

df_40550 = fetch_daily_quotes("40550", "2016-01-01", datetime.today().strftime("%Y-%m-%d"), ID_TOKEN)
if df_40550 is None:
    raise RuntimeError("40550 のデータ取得に失敗")

# 調べたいカラムリスト
cols = [
    "open","high","low","close",
    "adjusted_open","adjusted_high","adjusted_low","adjusted_close",
    "volume","turnover_value","adjusted_volume",
    "adjustment_factor"
]

for c in cols:
    if c in df_40550.columns:
        print(f"{c:18s} max = {df_40550[c].max()}")


open               max = 28400.0
high               max = 29260.0
low                max = 27950.0
close              max = 28920.0
adjusted_open      max = 7100.0
adjusted_high      max = 7315.0
adjusted_low       max = 6987.5
adjusted_close     max = 7230.0
volume             max = 2399500.0
turnover_value     max = 63491355000.0
adjusted_volume    max = 9598000.0
adjustment_factor  max = 1.0


In [15]:
from datetime import datetime

# PostgreSQL signed BIGINT の最大値
BIGINT_MAX = 2**63 - 1

# 調べたいコードのリスト（例として一括取得用の codes 変数がある想定）
for code in codes:
    df = fetch_daily_quotes(code, "2016-01-01", datetime.today().strftime("%Y-%m-%d"), ID_TOKEN)
    if df is None or df.empty:
        continue

    # 各カラムの最大値
    vv_max = df["volume"].max()              # float
    tv_max = df["turnover_value"].max()      # float
    av_max = df["adjusted_volume"].max()     # float

    # 範囲オーバーの候補をリスト化
    errs = []
    if pd.notna(vv_max) and vv_max > BIGINT_MAX:
        errs.append(f"volume({vv_max})")
    if pd.notna(tv_max) and tv_max > BIGINT_MAX:
        errs.append(f"turnover_value({tv_max})")
    if pd.notna(av_max) and av_max > BIGINT_MAX:
        errs.append(f"adjusted_volume({av_max})")

    if errs:
        print(f"⚠️ {code} overflow:", ", ".join(errs))


In [16]:
print(df[["volume","turnover_value","adjusted_volume"]].isna().sum())

volume             1
turnover_value     1
adjusted_volume    1
dtype: int64
